# IMA Plotter – Demo

End-to-end walkthrough: load Excel data → explore → baseline subtraction → interactive plots.

In [ ]:
# Cell 1 – Setup & Imports
import os
import sys

# Make sure the package is importable when running the notebook in-place
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from ima_plotter import load_excel_files, subtract_baseline, plot_magnetic_vs_time

SAMPLE_DIR = os.path.join("..", "sample_data")

In [ ]:
# Cell 2 – Load Data
df = load_excel_files(SAMPLE_DIR)

print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Cell 3 – Explore Data
print("IDs       :", sorted(df["id"].unique()))
print("Groups    :", sorted(df["group"].unique()))
print("Frequencies:", sorted(df["frequency"].unique()))
print()
df[["avg_v2abs_magnetic", "std_v2abs_magnetic", "avg_time"]].describe()

In [ ]:
# Cell 4 – Baseline Subtraction
# Subtract Cal0 from every group within each id + frequency combination.
# Combined uncertainty: sqrt(std_original² + std_baseline²)
df_sub = subtract_baseline(df, baseline_group="Cal0")

# Before vs after for one representative slice
sample = df_sub[(df_sub["id"] == "EX1A") & (df_sub["frequency"] == 2)]
sample[["group", "avg_time", "avg_v2abs_magnetic", "delta_avg_v2abs_magnetic"]].head(8)

In [ ]:
# Cell 5 – Basic Plot
# Single frequency, single ID, all groups, with error bars
fig = plot_magnetic_vs_time(
    df_sub,
    filter_frequency=2,
    filter_id="EX1A",
    use_baseline_subtracted=True,
    show_error_bars=True,
)
fig.show()

In [ ]:
# Cell 6 – Facet by ID
# One subplot per sample ID, single frequency
fig = plot_magnetic_vs_time(
    df_sub,
    facet_by="id",
    filter_frequency=2,
    use_baseline_subtracted=True,
    show_error_bars=True,
)
fig.show()

In [ ]:
# Cell 7 – Facet by Frequency
# One subplot per frequency, single ID
fig = plot_magnetic_vs_time(
    df_sub,
    facet_by="frequency",
    filter_id="EX1A",
    use_baseline_subtracted=True,
    show_error_bars=True,
)
fig.show()

In [ ]:
# Cell 8 – Combined Faceting (ID × Frequency)
# Grid of subplots: rows = ID, columns = frequency
# Filter to a subset to keep the grid manageable
fig = plot_magnetic_vs_time(
    df_sub,
    facet_by=["id", "frequency"],
    filter_id=["EX1A", "EX1B"],
    filter_frequency=[2, 7],
    use_baseline_subtracted=True,
    show_error_bars=True,
)
fig.update_layout(height=600)
fig.show()

In [ ]:
# Cell 9 – Export Data
output_path = os.path.join("..", "processed_data.csv")
df_sub.to_csv(output_path, index=False)
print(f"Saved processed data to: {os.path.abspath(output_path)}")